# House Price — Regression Baselines

A reproducible baseline experiment following the shared 25-section assignment workflow.

## 01. Problem Definition

Predict the numeric house price per square-foot field and compare three regression models.

## 02. Import Libraries

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 03. Experiment Configuration

In [ ]:
EXPERIMENT_NAME = "ml_baseline"
TARGET = "Price (in rupees)"
NUMERIC_FEATURES = ["Carpet Area Numeric", "Bathroom Numeric", "Balcony Numeric", "BHK"]
CATEGORICAL_FEATURES = ["location", "Status", "Transaction", "Furnishing", "facing", "Ownership"]
SELECTED_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BEST_METRIC = "RMSE"

## 04. Paths

In [ ]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data" / "house_prices.csv").exists(): PROJECT_DIR = Path("project/house-price-prediction")
DATA_DIR, FIGURE_DIR, MODEL_DIR, OUTPUT_DIR = (PROJECT_DIR / x for x in ("data", "figures", "models", "outputs"))
for directory in (FIGURE_DIR, MODEL_DIR, OUTPUT_DIR): directory.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR.resolve())

## 05. Load Dataset

In [ ]:
df = pd.read_csv(DATA_DIR / "house_prices.csv")
print("Shape:", df.shape); display(df.head())

## 06. Data Understanding

In [ ]:
df.info(); display(df.describe(include="all").T)
print("Duplicate rows:", int(df.duplicated().sum()))
display(df.isna().sum().sort_values(ascending=False).head(12).to_frame("missing"))

## 07. Data Cleaning

In [ ]:
df = df.drop_duplicates().copy()
df = df[df[TARGET].notna() & (df[TARGET] > 0)].copy()

## 08. Feature Engineering

Parse numeric area/bathroom/balcony values and BHK from messy text. Parsing happens before splitting, but all learned imputing/encoding/scaling stays inside the pipelines.

In [ ]:
df["Carpet Area Numeric"] = pd.to_numeric(df["Carpet Area"].str.extract(r"([\d,.]+)")[0].str.replace(",", "", regex=False), errors="coerce")
df["Bathroom Numeric"] = pd.to_numeric(df["Bathroom"].str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")
df["Balcony Numeric"] = pd.to_numeric(df["Balcony"].str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")
df["BHK"] = pd.to_numeric(df["Title"].str.extract(r"(\d+(?:\.\d+)?)\s*BHK", flags=2)[0], errors="coerce")

## 09. Feature Selection

In [ ]:
X = df[SELECTED_FEATURES].copy(); y = df[TARGET].astype(float)
print("Rows:", len(X), "Features:", SELECTED_FEATURES); display(y.describe())

## 10. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 11. Preprocessing

In [ ]:
numeric_scaled = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
numeric_tree = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))])
linear_prep = ColumnTransformer([("num", numeric_scaled, NUMERIC_FEATURES), ("cat", categorical, CATEGORICAL_FEATURES)])
tree_prep = ColumnTransformer([("num", numeric_tree, NUMERIC_FEATURES), ("cat", categorical, CATEGORICAL_FEATURES)])

## 12. Define 3 ML Models

In [ ]:
models = {
 "Linear Regression": Pipeline([("prep", linear_prep), ("model", LinearRegression())]),
 "Decision Tree Regressor": Pipeline([("prep", tree_prep), ("model", DecisionTreeRegressor(max_depth=18, min_samples_leaf=5, random_state=RANDOM_STATE))]),
 "Random Forest Regressor": Pipeline([("prep", tree_prep), ("model", RandomForestRegressor(n_estimators=100, max_depth=22, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))]),
}
model_files = {"Linear Regression":"linear_regression.joblib", "Decision Tree Regressor":"decision_tree_regressor.joblib", "Random Forest Regressor":"random_forest_regressor.joblib"}

## 13. Train Models

In [ ]:
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train); predictions[name] = model.predict(X_test); print("Trained:", name)

## 14. Evaluation Metrics

In [ ]:
rows=[]
for name, pred in predictions.items():
    rows.append({"Model":name, "MAE":mean_absolute_error(y_test,pred), "RMSE":mean_squared_error(y_test,pred,squared=False), "R2":r2_score(y_test,pred)})

## 15. Model Comparison Table

In [ ]:
results=pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True); display(results)

## 16. Visualization

In [ ]:
for metric, filename, color in [("MAE","mae_comparison.png","steelblue"),("RMSE","rmse_comparison.png","darkorange"),("R2","r2_comparison.png","seagreen")]:
    plt.figure(figsize=(9,4)); sns.barplot(data=results,x="Model",y=metric,color=color); plt.title(f"{metric} comparison"); plt.xticks(rotation=10); plt.tight_layout(); plt.savefig(FIGURE_DIR/filename,dpi=160); plt.show()

## 17. Select Best Model

In [ ]:
best_name=results.iloc[0]["Model"]; best_model=models[best_name]; best_pred=predictions[best_name]
plt.figure(figsize=(6,6)); plt.scatter(y_test,best_pred,alpha=.15,s=10); bounds=[min(y_test.min(),best_pred.min()),max(y_test.max(),best_pred.max())]; plt.plot(bounds,bounds,"r--"); plt.xlabel("Actual price"); plt.ylabel("Predicted price"); plt.title(f"Actual vs predicted: {best_name}"); plt.tight_layout(); plt.savefig(FIGURE_DIR/"actual_vs_predicted.png",dpi=160); plt.show()
print("Best model by RMSE:",best_name)

## 18. Save 3 Models

In [ ]:
model_paths={}
for name,model in models.items():
    path=MODEL_DIR/model_files[name]; joblib.dump(model,path); model_paths[name]=path

## 19. Save Best Model

In [ ]:
best_path=MODEL_DIR/"best_house_price_model.joblib"; joblib.dump(best_model,best_path)

## 20. Save Results

In [ ]:
results.to_csv(OUTPUT_DIR/"house_price_ml_results.csv",index=False)

## 21. Save Feature/Config

In [ ]:
config={"experiment":EXPERIMENT_NAME,"task":"regression","target":TARGET,"features":SELECTED_FEATURES,"models":list(models),"best_metric":BEST_METRIC,"best_model":best_name,"random_state":RANDOM_STATE}
with open(OUTPUT_DIR/"model_config.json","w",encoding="utf-8") as f: json.dump(config,f,indent=2,ensure_ascii=False)

## 22. Reload 3 Models

In [ ]:
loaded_models={name:joblib.load(path) for name,path in model_paths.items()}

## 23. Predict Test Sample

In [ ]:
sample=X_test.iloc[[0]]; actual=float(y_test.iloc[0]); verification=[]
for name,model in loaded_models.items():
    pred=float(model.predict(sample)[0]); verification.append({"Model":name,"Predicted Price":pred,"Actual Price":actual,"Absolute Error":abs(pred-actual)})
display(pd.DataFrame(verification))

## 24. Verify Saved Models

In [ ]:
for name in models: np.testing.assert_allclose(loaded_models[name].predict(sample),models[name].predict(sample))
assert joblib.load(best_path).predict(sample).shape==(1,)
print("[OK] All saved models loaded and predicted successfully.")

## 25. Conclusion

In [ ]:
print(f"Best model: {best_name} | RMSE={results.iloc[0]['RMSE']:.2f} | R²={results.iloc[0]['R2']:.4f}")
print("Saved deployable preprocessing+model pipelines; lower MAE/RMSE and higher R² are better.")